Task 1: Pipeline Architecture Diagram 
1.1 — Draw the end-to-end architecture


                         DATA SOURCES
                ┌──────────────────────────┐
                │ Historical Batch Dataset │
                │ (Online Retail .xlsx)    │
                └──────────────┬───────────┘
                               │
                               │ Batch Upload
                               │
                               ▼
                        ┌───────────────┐
                        │ Batch Ingest  │
                        │  (File Loader)│
                        └───────┬───────┘


                ┌──────────────────────────┐
                │ Live Transaction Stream  │
                │ (Row-by-row purchases)   │
                └──────────────┬───────────┘
                               │
                               │ Stream Events
                               │
                               ▼
                       ┌───────────────┐
                       │ Stream Ingest │
                       │ (Event Queue) │
                       └───────┬───────┘
                               │
                               ▼
                     ┌─────────────────────┐
                     │   Landing Zone      │
                     │   Raw Data Storage  │
                     │  (Unprocessed Data) │
                     └─────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │  Validation Stage    │
                    │ Schema + Quality     │
                    │ Checks               │
                    └───────┬──────────────┘
                            │
                ┌───────────┴───────────┐
                ▼                       ▼
       ┌──────────────────┐    ┌───────────────────┐
       │ Valid Records     │    │ Quarantine / Dead │
       │                   │    │ Letter Storage    │
       │                   │    │ (Failed Records)  │
       └─────────┬─────────┘    └───────────────────┘
                 │
                 ▼
        ┌──────────────────────────┐
        │ Transformation Layer     │
        │ Cleaning + Enrichment    │
        │ Feature Engineering      │
        └──────────┬───────────────┘
                   │
                   ▼
            STORAGE LAYERS
   ┌─────────────────────────────────┐
   │ Raw Layer (immutable storage)  │
   │ Format: Parquet                │
   └──────────────┬──────────────────┘
                  ▼
   ┌─────────────────────────────────┐
   │ Clean Layer (validated data)   │
   │ Format: Partitioned Parquet    │
   └──────────────┬──────────────────┘
                  ▼
   ┌─────────────────────────────────┐
   │ Feature Layer                   │
   │ ML-ready datasets               │
   │ Format: Feature tables         │
   └──────────────┬──────────────────┘
                  │
                  ▼
            DATA CONSUMERS
        ┌──────────────┴──────────────┐
        ▼                             ▼
 ┌───────────────┐             ┌────────────────┐
 │ BI Dashboard  │             │ ML Model       │
 │ (Analytics)   │             │ Training /     │
 │               │             │ Prediction     │
 └───────────────┘             └────────────────┘


                     MONITORING & ALERTING
            ┌──────────────────────────────────┐
            │ Data Quality Checks              │
            │ Pipeline Metrics                 │
            │ Alerts on failures/anomalies     │
            └──────────────────────────────────┘

1.2 Component Descriptions

1. Data Sources

The pipeline begins with two main data sources: a historical batch dataset and a live transaction stream. The batch dataset contains historical retail transactions stored in an Excel file (.xlsx), which is typically used for initial data loading and backfilling historical records. The live stream represents real-time purchase events generated continuously from an online retail system. These sources provide the raw transactional data needed for analytics and machine learning. Their outputs are raw transaction records that enter the ingestion layer.


2. Batch Ingestion Layer

The batch ingestion component is responsible for loading historical data files into the pipeline. It reads structured files such as Excel or CSV and converts them into a standardized internal format for processing. This process may run on a schedule, such as daily or weekly, depending on how often historical files are updated. The output of this stage is raw transaction data ready to be stored in the landing zone. Technologies such as Python scripts, Airflow jobs, or ETL tools can be used here.

3. Stream Ingestion Layer

The stream ingestion component handles real-time transaction data as it is generated by the system. Each transaction event is pushed into a streaming platform that buffers and distributes the data for downstream processing. This allows the pipeline to process events continuously instead of waiting for batch files. The output of this stage is a stream of raw event records. Technologies such as Apache Kafka, Kinesis, or message queues are commonly used for streaming ingestion.

4. Landing Zone (Raw Storage)

The landing zone is the first storage layer where raw data is stored exactly as it was received. No transformations or cleaning are applied at this stage, ensuring the original data is preserved for auditing and debugging purposes. Both batch and streaming data are written into this storage area. Data is typically stored in a scalable storage system such as a data lake using Parquet files partitioned by date. This layer acts as a permanent archive of all incoming records.

5. Validation Stage

The validation stage ensures that incoming records meet predefined data quality and schema requirements. This includes checking whether required fields exist, verifying data types, validating numerical ranges, and detecting missing values. Records that pass validation continue to the transformation stage, while records that fail are redirected to a quarantine area. The output of this stage is two streams of data: validated records and rejected records. Tools such as Great Expectations, custom Python validation scripts, or Spark validation jobs may be used here.

6. Quarantine / Dead Letter Area

The quarantine area stores records that fail validation checks. Instead of discarding invalid data, these records are preserved so that engineers can inspect and fix them later. This helps maintain data integrity and allows debugging of upstream issues. Each record typically includes an error reason explaining why it failed validation. The output is a collection of rejected records stored separately from the main dataset. This storage may be implemented as a separate folder in the data lake or a dedicated error table.

7. Transformation Stage

The transformation layer processes validated data to prepare it for analytics and machine learning. This stage includes cleaning missing values, standardizing formats, enriching records with additional attributes, and generating new features. For example, new columns such as total purchase value or customer lifetime metrics may be computed. The output of this stage is structured, enriched datasets ready for storage in analytical layers. Tools such as Apache Spark, Pandas pipelines, or dbt transformations are typically used.

8. Storage Layers

The pipeline organizes processed data into multiple storage layers following the medallion architecture (raw, clean, feature).

The raw layer stores the original ingested data in an immutable format for traceability and recovery. The clean layer contains validated and standardized data that has passed quality checks and is ready for analytics. The feature layer contains engineered features specifically designed for machine learning models. These datasets are often stored in Parquet format with date partitioning to optimize query performance and scalability.

9. Consumer Interfaces

The final datasets are accessed by two primary consumers: a business intelligence dashboard and a machine learning model. The BI dashboard queries the clean data layer to generate reports, metrics, and visualizations for business users. The ML model uses the feature layer to train predictive models and generate predictions. Access to these datasets is typically provided through SQL query engines, APIs, or data warehouse connections. These interfaces allow downstream systems to efficiently consume processed data.

10. Monitoring and Alerting

Monitoring ensures that the pipeline operates reliably and maintains data quality over time. Metrics such as pipeline runtime, record counts, validation failure rates, and processing latency are continuously tracked. If anomalies occur, such as sudden spikes in failed records or pipeline crashes, alerts are triggered to notify engineers. Monitoring systems may use tools like Prometheus, Grafana, or built-in orchestration alerts. This layer helps maintain the health, reliability, and observability of the entire pipeline.

Task 2: Validation and Error Handling Design 

2.1 Validation Rules

The pipeline must validate incoming records to ensure data quality and consistency before transformation. Validation rules are grouped into three categories: schema validations, value range validations, and business rule validations.

1. Schema Validations (Structural Correctness)

Schema validation ensures that each record follows the expected structure and data types

| Field       | Expected Type | Required | Validation Rule                           |
| ----------- | ------------- | -------- | ----------------------------------------- |
| InvoiceNo   | String        | Yes      | Must be a non-empty string                |
| StockCode   | String        | Yes      | Must be a non-empty string                |
| Description | String        | No       | May be null but must be string if present |
| Quantity    | Integer       | Yes      | Must be an integer value                  |
| InvoiceDate | Datetime      | Yes      | Must be a valid timestamp                 |
| UnitPrice   | Float         | Yes      | Must be a numeric value                   |
| CustomerID  | Integer       | No       | Must be numeric if present                |
| Country     | String        | Yes      | Must be a non-empty string                |




Example Schema Rules

InvoiceNo must be a non-empty string

StockCode must be a non-empty string

Quantity must be an integer

UnitPrice must be numeric

InvoiceDate must be a valid datetime

Country must not be null

If any of these validations fail, the record is considered structurally invalid.

2. Value Range Validations (Sensible Values)

Value validations ensure that numerical values fall within realistic ranges.

Rules

UnitPrice must be greater than 0 for normal transactions

Quantity should not be equal to 0

UnitPrice must be less than a defined upper limit (e.g., < 10000) to detect data entry errors

Quantity should be within a reasonable range (e.g., -10000 to 10000)

InvoiceDate cannot be in the future

These checks help detect data entry mistakes, corrupted values, or unrealistic transactions.

3. Business Rule Validations (Domain Logic)

Business rules ensure consistency between fields and enforce domain-specific logic.

Rules

If InvoiceNo starts with "C" (cancellation), then Quantity must be negative

If Quantity is negative, InvoiceNo must represent a cancellation

Total transaction value (Quantity × UnitPrice) should not be zero

CustomerID should exist for repeat customers in historical data

StockCode must correspond to a valid product in the product catalog

These validations help enforce business-level correctness rather than just structural correctness.

2.2 - Design the error handling flow 
Different types of validation failures require different error handling strategies to maintain pipeline reliability while preserving problematic data for debugging.

1. Schema Validation Failures
What happens?

Records that fail schema validation are rejected entirely, because the pipeline cannot safely process records with incorrect structure or types.

Where do they go?

Failed records are sent to a Dead Letter Queue (DLQ) or quarantine table in the raw storage layer.

The stored record includes:

original raw data

timestamp

validation error message

pipeline stage identifier

Operator Alerts

If the number of schema failures exceeds a defined threshold, a monitoring system triggers alerts through:

pipeline logs

monitoring dashboards

messaging alerts (email or Slack)

Recovery Strategy

Operators inspect quarantined records, identify the root cause (e.g., incorrect schema in source file), fix the upstream data source, and reprocess the quarantined data through the pipeline.

2. Value Range Validation Failures
What happens?

Records with invalid numeric ranges are typically flagged rather than fully rejected, because the structure is valid but the values may contain errors.

Where do they go?

The records are stored in a quarantine dataset with an error label indicating which validation rule failed.

Operator Alerts

Monitoring tools track spikes in value validation failures, which may indicate upstream issues such as pricing errors or corrupted input streams.

Recovery Strategy

After identifying the issue, operators can either:

correct the values

reload the source data

or re-run the transformation pipeline on corrected records.

3. Business Rule Validation Failures
What happens?

Records failing business logic checks are accepted but flagged for review, because they may represent legitimate but unusual events.

Where do they go?

These records are stored in a review table with flags indicating which business rule failed.

Operator Alerts

Alerts are triggered if unusual patterns occur, such as a large number of cancellations or negative quantities.

Recovery Strategy

Data analysts review flagged records and determine whether they represent:

legitimate business activity

system errors

or fraud/anomalies.

After verification, the records can either be corrected or approved for inclusion in the clean dataset.

Task 3: Transformation and Storage Design

3.1 Define Transformations
After records pass the validation stage, they enter the transformation layer, where the data is cleaned, enriched, and converted into analytics-ready datasets. Each transformation must be deterministic and preferably idempotent, meaning it can be safely re-run without producing incorrect results.

1. Data Type Standardization

Input:
Validated raw transaction records containing fields such as InvoiceNo, StockCode, Quantity, InvoiceDate, UnitPrice, CustomerID, and Country.

Transformation:
Convert all fields to standardized types:

InvoiceDate → datetime

Quantity → integer

UnitPrice → float

CustomerID → integer

Output:
A dataset with consistent and correctly typed fields.

Idempotency:
Yes. Re-running the conversion results in the same dataset since type conversions are deterministic.

2. Handling Missing Values

Input:
Validated dataset that may still contain optional missing values such as Description or CustomerID.

Transformation:

Replace missing Description with "Unknown Product".

Keep missing CustomerID as NULL but mark them as guest transactions.

Output:
Cleaned dataset where optional fields have consistent representations.

Idempotency:
Yes. Re-running the transformation produces the same result because the same replacement rule is applied.

3. Duplicate Removal

Input:
Validated transaction dataset that may contain duplicate rows caused by ingestion retries.

Transformation:
Remove duplicates using a unique key such as:

InvoiceNo + StockCode + CustomerID + InvoiceDate

Output:
A dataset where each transaction record is unique.

Idempotency:
Yes. Removing duplicates repeatedly does not change the result after the first run.

4. Derived Column: Line Total

Input:
Clean transaction records containing Quantity and UnitPrice.

Transformation:

LineTotal = Quantity × UnitPrice

Output:
Transaction dataset with an additional column:

LineTotal

representing the total value of each line item.

Idempotency:
Yes. The value is always computed from the same source columns.

5. Derived Column: Cancellation Flag

Input:
Transaction records containing InvoiceNo.

Transformation:

Create a boolean flag:

IsCancellation = InvoiceNo starts with "C"

Output:
Dataset with a new column:

IsCancellation (True/False)

Idempotency:
Yes. The rule always produces the same result when re-run.

6. Date Feature Extraction

Input:
Transaction records containing InvoiceDate.

Transformation:
Extract useful date components:

InvoiceYear

InvoiceMonth

InvoiceDay

DayOfWeek

HourOfDay

Output:
Dataset enriched with time-based features useful for analytics.

Idempotency:
Yes. These features are deterministically derived from the timestamp

7. Customer-Level Revenue Aggregation
Input:
Clean transaction dataset containing CustomerID and LineTotal.

Transformation:

Aggregate per customer:

TotalRevenue = SUM(LineTotal)
Output:
Customer-level dataset with total spending.

Example:

CustomerID	TotalRevenue
12345	1500.50
Idempotency:
Yes, provided the aggregation always runs over the same source dataset.



8. Customer Order Count

Input:
Clean transaction dataset.

Transformation:

Calculate number of unique orders per customer:

OrderCount = COUNT(DISTINCT InvoiceNo)

Output:
Customer table with order frequency information.

Idempotency:
Yes. Aggregation over the same dataset yields identical results.

9. Product Diversity per Customer

Input:
Transaction dataset containing CustomerID and StockCode.

Transformation:

Calculate the number of unique products purchased by each customer:

ProductDiversity = COUNT(DISTINCT StockCode)

Output:
Customer-level metric indicating purchasing variety.

Idempotency:
Yes. Deterministic aggregation ensures safe re-runs.

10. Recency Feature

Input:
Transaction dataset containing InvoiceDate.

Transformation:

Compute recency as the number of days since the last purchase:

Recency = CurrentDate − MAX(InvoiceDate per customer)

Output:
Customer feature showing how recently they made a purchase.

Idempotency:
Yes if the reference date (observation date) is fixed during the feature generation process.

11. ML Feature Engineering (Observation Window)

Input:
Customer transaction history within a defined observation window (e.g., the past 6 months).

Transformation:

Create machine learning features such as:

CustomerRevenue_6M

OrderFrequency_6M

AverageBasketSize

ProductDiversity_6M

RecencyDays

Only transactions within the observation window are used to prevent data leakage.

Output:
A structured feature table used for training ML models.

Example:

CustomerID	Revenue_6M	OrderCount_6M	ProductDiversity	RecencyDays

Idempotency:
Yes if the observation window and reference date are fixed.

3.2 Design the Storage Layers

A layered storage architecture separates raw, processed, and machine-learning-ready data. This improves data reliability, scalability, and reusability while allowing different consumers to access the appropriate level of data processing.

| Layer   | Contents                                                                                                     | Format                                                  | Update Frequency                                      | Retention                                       |
| ------- | ------------------------------------------------------------------------------------------------------------ | ------------------------------------------------------- | ----------------------------------------------------- | ----------------------------------------------- |
| Raw     | Original ingested batch and streaming transaction data without transformations                               | Parquet files partitioned by date                       | Continuous ingestion / daily batch                    | Long-term (full historical archive)             |
| Clean   | Validated and transformed transaction data with derived fields (LineTotal, date features, cancellation flag) | Partitioned Parquet or Delta tables                     | Updated incrementally as new validated records arrive | Medium to long-term                             |
| Feature | Aggregated customer-level features used for machine learning models                                          | Feature tables (Parquet / Delta / feature store format) | Periodic refresh (daily or weekly)                    | Limited retention based on model training needs |


Raw Layer

The raw layer stores the original ingested data exactly as it was received from the sources. No transformations or cleaning are applied in this layer to preserve the original records for auditing and debugging purposes.

The data is stored in Parquet format partitioned by ingestion date because it provides efficient compression and fast scanning for large datasets. Since this layer may grow very large over time, columnar storage reduces storage costs and improves performance. The raw layer is typically read only by data engineers or pipeline processes when reprocessing data or investigating errors.

Clean Layer

The clean layer contains validated and transformed datasets after applying the transformations defined in section 3.1. This includes standardized data types, derived columns such as LineTotal, date features, and cleaned records that passed validation rules.

This layer is stored as partitioned Parquet or Delta tables, which allow efficient queries and incremental updates. Partitioning by date (for example by InvoiceDate) improves query performance for analytics workloads. The clean layer is accessed by BI tools, analytics queries, and downstream data transformations, making it the primary analytical dataset.

Feature Layer

The feature layer contains aggregated datasets designed specifically for machine learning models. These tables include customer-level metrics such as total revenue, order count, product diversity, and recency within a defined observation window.

This layer can be stored as feature tables using Parquet or Delta format, or managed by a dedicated feature store. Feature datasets are typically refreshed on a scheduled basis, such as daily or weekly, depending on the model retraining schedule. Retention may be limited because older feature snapshots may no longer be needed after model training.

Format Justification

Parquet is chosen for most storage layers because it is a columnar storage format optimized for analytics workloads. It provides high compression rates, efficient query performance, and compatibility with many data processing engines.

Delta tables may be used in the clean and feature layers because they support ACID transactions, schema evolution, and time-travel queries, allowing engineers to read previous versions of the data if errors occur. Time-travel capabilities are particularly useful when debugging pipelines or reproducing historical model training datasets.

3.3 Incremental Update Strategy

To support continuous data ingestion and scalable processing, the pipeline must handle new data incrementally rather than reprocessing the entire dataset.

Tracking Processed Data

The pipeline tracks which records have already been processed using a high-water mark strategy. The high-water mark represents the latest processed value of a timestamp field such as InvoiceDate or an ingestion timestamp.

Each pipeline run only processes records that have timestamps greater than the previously recorded high-water mark. After processing completes successfully, the high-water mark is updated to reflect the latest processed record. This prevents duplicate processing and improves efficiency.

Handling Late-Arriving Records

Late-arriving records may occur due to delayed data transmission or ingestion issues. To handle this scenario, the pipeline processes data using a sliding time window.

For example, each run may reprocess the last few days of data to capture any delayed records. The pipeline also performs deduplication using unique transaction keys to ensure that previously processed records are not duplicated when reprocessing overlapping time windows.

Refreshing Customer-Level Features

Customer-level features are refreshed periodically using incremental aggregations. When new transactions arrive, only the affected customers need to have their features recalculated.

The pipeline identifies customers associated with newly processed transactions and recomputes their metrics such as revenue, order count, and recency. This approach avoids recomputing features for all customers and significantly reduces processing costs. Feature tables are typically updated on a daily schedule or during model retraining cycles.